# 🧠 Model Training & Evaluation - LogisChain AI

This notebook orchestrates the training and evaluation of all 5 core model architectures:
1. **HetGAT** - Heterogeneous Graph Attention Network for risk propagation
2. **TCN** - Temporal Convolutional Network for demand/throughput forecasting
3. **Transformer** - Shipment Risk Encoder for delay/damage prediction
4. **XGBoost** - Tabular baseline with Optuna + SHAP explainability
5. **Survival** - Cox PH and DeepSurv for time-to-default modeling
6. **Ensemble** - LightGBM stacking meta-learner

In [ ]:
import os
import sys
import yaml
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is on path
sys.path.insert(0, os.path.abspath('..'))

with open('../configs/data_config.yaml') as f:
    data_cfg = yaml.safe_load(f)
with open('../configs/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

print('Configs loaded successfully.')

## 1. Train HetGAT Graph Neural Network

In [ ]:
from src.models.gnn import SupplyChainGraph, train_gnn

builder = SupplyChainGraph(
    features_path=os.path.join('..', data_cfg['paths']['features_dir'], 'master_features.csv'),
    raw_dir=os.path.join('..', data_cfg['paths']['raw_dir'])
)
graph_data = builder.build()
print(f"Graph: {graph_data['meta']['num_suppliers']} suppliers, "
      f"{graph_data['meta']['num_mfrs']} manufacturers, "
      f"{graph_data['meta']['num_ports']} ports")

gnn_model = train_gnn(graph_data, model_cfg['gnn'], output_dir='../data/models')

## 2. Train TCN Forecaster

In [ ]:
from src.models.tcn import train_tcn

tcn_model = train_tcn(
    model_cfg['tcn'],
    raw_dir=os.path.join('..', data_cfg['paths']['raw_dir']),
    output_dir='../data/models'
)

## 3. Train Transformer Shipment Risk Encoder

In [ ]:
from src.models.transformer import train_transformer

transformer_model = train_transformer(
    model_cfg['transformer'],
    raw_dir=os.path.join('..', data_cfg['paths']['raw_dir']),
    output_dir='../data/models'
)

## 4. Train XGBoost + Optuna + SHAP

In [ ]:
from src.models.xgboost_model import train_xgboost

xgb_model = train_xgboost(
    model_cfg['xgboost'],
    features_dir=os.path.join('..', data_cfg['paths']['features_dir']),
    output_dir='../data/models',
    n_trials=100
)

## 5. Train Survival Models (Cox PH + DeepSurv)

In [ ]:
from src.models.survival import train_coxph, train_deepsurv

cph_model, c1 = train_coxph(
    features_dir=os.path.join('..', data_cfg['paths']['features_dir']),
    output_dir='../data/models'
)
print(f'\nCox PH C-index: {c1:.4f}')

ds_model, c2 = train_deepsurv(
    features_dir=os.path.join('..', data_cfg['paths']['features_dir']),
    output_dir='../data/models'
)
print(f'DeepSurv C-index: {c2:.4f}')

## 6. Train Stacking Ensemble

In [ ]:
from src.models.ensemble import train_ensemble

ensemble_model = train_ensemble(
    config=model_cfg.get('ensemble', {}),
    features_dir=os.path.join('..', data_cfg['paths']['features_dir']),
    models_dir='../data/models',
    output_dir='../data/models'
)

## 📊 Results Summary

In [ ]:
print('\n' + '='*60)
print('Phase 2 Model Training Complete')
print('='*60)
print(f'Models saved to: ../data/models/')
print(f'MLflow experiments logged for all architectures.')
print(f'\nSaved model artifacts:')
for f in os.listdir('../data/models'):
    size = os.path.getsize(os.path.join('../data/models', f))
    print(f'  {f:<40} {size/1024:.1f} KB')